# Evaluación — Mantenimiento Predictivo

Este notebook reproduce el flujo de trabajo del curso: exploración de datos, ingeniería de variables, entrenamiento de un modelo de clasificación y evaluación de resultados.

**Instrucciones generales**
- Ejecuta las celdas en orden.
- Donde veas `___`, escribe el valor o expresión que corresponda.
- Las celdas de texto (como esta) no necesitan modificación.
- Al final hay preguntas de interpretación — respóndelas en la celda de texto correspondiente.

## Preparación del ambiente — leer antes de ejecutar

Este notebook requiere **Python 3.11**. Si tienes varias versiones instaladas (por ejemplo 3.11 y 3.14), debes indicar explícitamente cuál usar al crear el ambiente virtual.

---

### Windows

Abre una terminal (PowerShell o CMD) en la carpeta raíz del repositorio y ejecuta:

```
py -3.11 -m venv venv
venv\Scripts\activate
pip install -r requierements.txt
```

Una vez instaladas las dependencias, abre Jupyter:

```
jupyter notebook
```

---

### Mac / Linux

```bash
python3.11 -m venv venv
source venv/bin/activate
pip install -r requierements.txt
jupyter notebook
```

---

### Verificación

Cuando el ambiente esté activo, verás `(venv)` al inicio de la línea en tu terminal. Confirma la versión de Python con:

```
python --version
```

Debe mostrar `Python 3.11.x`. Si muestra otra versión, el ambiente no se creó con 3.11 — borra la carpeta `venv` y repite el proceso.

> Si ya tienes el ambiente del curso activado, no es necesario volver a crearlo.

## 1. Importaciones y configuración

Ejecuta esta celda sin modificarla. Carga todas las librerías que se usarán a lo largo del notebook.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve
)
from sklearn.inspection import permutation_importance

TEMPLATE = 'plotly_white'
DATA_DIR = Path('data')
RANDOM_STATE = 42

## 2. Carga de datos

El dataset tiene 10,000 observaciones de un proceso industrial. Cada fila es un punto de operación con sus condiciones de sensor y el estado de fallo resultante.

In [ ]:
df = pd.read_csv(DATA_DIR / 'predictive_maintenance.csv', encoding='utf-8-sig')

df = df.rename(columns={
    'UDI': 'id',
    'Product ID': 'product_id',
    'Type': 'machine_type',
    'Air temperature [K]': 'air_temp',
    'Process temperature [K]': 'process_temp',
    'Rotational speed [rpm]': 'speed',
    'Torque [Nm]': 'torque',
    'Tool wear [min]': 'tool_wear',
    'Target': 'target',
    'Failure Type': 'failure_type'
})

# El target correcto se construye desde failure_type
df['failure'] = (df['failure_type'] != 'No Failure').astype(int)

print(f"Dimensiones: {df.shape}")
df.head()

## 3. Desbalance de clases

Antes de entrenar cualquier modelo, hay que entender qué tan frecuentes son los fallos en el dataset.

In [ ]:
# --- EJERCICIO 1 ---
# Calcula el porcentaje de fallos en el dataset.
# Pista: .mean() sobre una columna binaria devuelve la proporción de 1s.

tasa_fallos = df['failure']._____()
print(f"Tasa de fallos: {tasa_fallos * 100:.2f}%")

In [ ]:
# Distribución visual del target — celda dada, solo ejecuta
counts = df['failure'].value_counts().reset_index()
counts.columns = ['failure', 'n']
counts['label'] = counts['failure'].map({0: 'No failure', 1: 'Failure'})

fig = px.bar(
    counts, x='label', y='n', color='label',
    color_discrete_map={'No failure': 'steelblue', 'Failure': 'tomato'},
    title='Distribución del target',
    labels={'n': 'Observaciones', 'label': ''},
    template=TEMPLATE
)
fig.show()

## 4. Variables derivadas

Los mecanismos físicos de fallo no siempre se capturan directamente con los sensores disponibles. A partir de la teoría del proceso se construyen tres variables nuevas:

| Variable | Fórmula | Mecanismo que captura |
|---|---|---|
| `power` | τ × (ω × 2π/60) [W] | Power Failure: potencia fuera de rango |
| `temp_diff` | T_proceso − T_aire [K] | Heat Dissipation Failure: diferencial de temperatura insuficiente |
| `strain` | torque × tool_wear | Overstrain Failure: carga alta con herramienta desgastada |

In [ ]:
# --- EJERCICIO 2 ---
# Escribe las tres líneas que calculan las variables derivadas.
# Usa la tabla de la sección anterior como referencia.
# Nota: la velocidad angular en rad/s se obtiene de rpm con la conversión ω = rpm × 2π / 60.

df['power']     = _____
df['temp_diff'] = _____
df['strain']    = _____

print(df[['power', 'temp_diff', 'strain']].describe().round(2))

In [ ]:
# Distribución de variables derivadas por estado de fallo — celda dada, solo ejecuta
DERIVED = ['power', 'temp_diff', 'strain']
labels = {0: 'No failure', 1: 'Failure'}
colors = {0: 'steelblue', 1: 'tomato'}

fig = make_subplots(rows=1, cols=3, subplot_titles=DERIVED)

for i, col in enumerate(DERIVED):
    for val in [0, 1]:
        data = df[df['failure'] == val][col]
        fig.add_trace(
            go.Histogram(
                x=data, name=labels[val],
                marker_color=colors[val],
                opacity=0.6, showlegend=(i == 0)
            ),
            row=1, col=i + 1
        )

fig.update_layout(
    barmode='overlay', template=TEMPLATE,
    title='Variables derivadas — sin fallo vs fallo',
    height=400
)
fig.show()

## 5. Preparación de datos para el modelo

Se separa el dataset en entrenamiento y prueba. La proporción estándar es 80/20. El parámetro `stratify=y` asegura que la tasa de fallos sea similar en ambas partes, lo cual es importante cuando hay desbalance.

In [ ]:
# --- EJERCICIO 3 ---
# Completa los argumentos del split.
# test_size controla qué fracción va a prueba (0.2 = 20%).
# stratify asegura que la proporción de fallos sea igual en train y test.

INPUT_FEATURES = ['air_temp', 'process_temp', 'speed', 'torque', 'tool_wear', 'machine_type']
X = df[INPUT_FEATURES]
y = df['failure']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=_____,
    stratify=_____,
    random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Tasa de fallos — train: {y_train.mean():.4f}  |  test: {y_test.mean():.4f}")

## 6. Pipeline de preprocesamiento

El pipeline encadena tres pasos en un solo objeto:

1. **`FeatureEngineer`** — calcula `power`, `temp_diff` y `strain` a partir de los sensores
2. **`ColumnTransformer`** — escala las variables numéricas (`StandardScaler`) y codifica `machine_type` (`OrdinalEncoder`)
3. **Clasificador** — el modelo que se entrena sobre los datos ya transformados

La ventaja del pipeline es que el preprocesamiento se aplica automáticamente tanto en entrenamiento como en predicción.

In [ ]:
# Definición del transformador de features — celda dada, solo ejecuta
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=INPUT_FEATURES)
        X['power']     = X['torque'] * (X['speed'] * 2 * np.pi / 60)
        X['temp_diff'] = X['process_temp'] - X['air_temp']
        X['strain']    = X['torque'] * X['tool_wear']
        return X


NUM_COLS = ['air_temp', 'process_temp', 'speed', 'torque', 'tool_wear',
            'power', 'temp_diff', 'strain']
CAT_COLS = ['machine_type']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), NUM_COLS),
    ('cat', OrdinalEncoder(categories=[['H', 'M', 'L']]), CAT_COLS)
], remainder='drop')

print("FeatureEngineer y preprocessor definidos.")

In [ ]:
# --- EJERCICIO 4 ---
# Construye el pipeline y entrénalo.
# El clasificador ya está instanciado abajo.
# Necesitas:
#   a) Armar el pipeline con los tres pasos: 'eng', 'pre', 'clf'
#   b) Llamar .fit() con los datos de entrenamiento

clasificador = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    n_jobs=-1,
    random_state=RANDOM_STATE
)

pipeline = Pipeline([
    ('eng', _____),
    ('pre', preprocessor),
    ('clf', _____)
])

pipeline._____(X_train, y_train)
print("Entrenamiento completo.")

## 7. Evaluación del modelo

Se evalúa sobre el conjunto de prueba, que el modelo no vio durante el entrenamiento.

In [ ]:
# Métricas — celda dada, solo ejecuta
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred,
                             target_names=['No failure', 'Failure'], digits=3))

auc_roc = roc_auc_score(y_test, y_prob)
avg_prec = average_precision_score(y_test, y_prob)
print(f"AUC-ROC:       {auc_roc:.4f}")
print(f"Avg Precision: {avg_prec:.4f}")

In [ ]:
# Curva ROC — celda dada, solo ejecuta
fpr, tpr, _ = roc_curve(y_test, y_prob)

fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines',
                         name=f'Random Forest (AUC={auc_roc:.3f})'))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines',
                         line=dict(dash='dash', color='gray'),
                         name='Clasificador aleatorio'))
fig.update_layout(
    title='Curva ROC',
    xaxis_title='Tasa de Falsos Positivos (FPR)',
    yaxis_title='Tasa de Verdaderos Positivos (TPR)',
    template=TEMPLATE
)
fig.show()

In [ ]:
# Curva Precision-Recall — celda dada, solo ejecuta
prec, rec, _ = precision_recall_curve(y_test, y_prob)
baseline = y_test.mean()

fig = go.Figure()
fig.add_trace(go.Scatter(x=rec, y=prec, mode='lines',
                         name=f'Random Forest (AP={avg_prec:.3f})'))
fig.add_hline(y=baseline, line_dash='dash', line_color='gray',
              annotation_text=f'baseline ({baseline:.3f})')
fig.update_layout(
    title='Curva Precision-Recall',
    xaxis_title='Recall',
    yaxis_title='Precision',
    template=TEMPLATE
)
fig.show()

## 8. Importancia de features

Permutation importance mide cuánto cae el Avg Precision cuando se permuta aleatoriamente cada variable de entrada. Una caída grande indica que el modelo depende de esa variable; una caída cercana a cero indica que la variable es irrelevante para el modelo.

In [ ]:
# Importancia de features — celda dada, solo ejecuta
perm = permutation_importance(
    pipeline, X_test, y_test,
    n_repeats=10, random_state=RANDOM_STATE,
    scoring='average_precision'
)

imp_df = pd.DataFrame({
    'feature': INPUT_FEATURES,
    'importance': perm.importances_mean,
    'std': perm.importances_std
}).sort_values('importance', ascending=False)

fig = px.bar(
    imp_df, x='feature', y='importance', error_y='std',
    title='Importancia de features (Permutation Importance)',
    labels={'feature': 'Variable', 'importance': 'Caída en Avg Precision'},
    template=TEMPLATE
)
fig.show()
print(imp_df.to_string(index=False))

## 9. Predicción sobre un caso nuevo

Un técnico reporta la siguiente lectura de un equipo tipo H al final de un turno:

| Variable | Valor |
|---|---|
| `air_temp` | 305.0 K |
| `process_temp` | 314.5 K |
| `speed` | 1180 rpm |
| `torque` | 71.5 Nm |
| `tool_wear` | 210 min |
| `machine_type` | H |

El pipeline ya entrenado puede recibir esta observación directamente — no es necesario calcular las variables derivadas a mano, el pipeline las calcula internamente.

In [ ]:
# --- EJERCICIO 5 ---
# a) Construye un DataFrame con la observación del técnico.
# b) Usa el pipeline para obtener la probabilidad de fallo.
# c) Responde en la celda de texto siguiente: ¿recomendarías una intervención?

nueva_obs = pd.DataFrame([{
    'air_temp': _____,
    'process_temp': _____,
    'speed': _____,
    'torque': _____,
    'tool_wear': _____,
    'machine_type': '_____'
}])

prob_fallo = pipeline.predict_proba(_____)[0, 1]
print(f"Probabilidad de fallo: {prob_fallo:.4f}")

**¿Recomendarías una intervención de mantenimiento para este equipo? Justifica con base en la probabilidad obtenida y en los valores de las variables de entrada.**

*Escribe tu respuesta aquí.*

## 10. Preguntas de interpretación

Responde en las celdas de texto. Donde se pide un número, cítalo directamente del output de tu ejecución.

### Pregunta 1

El dataset tiene ~3.6% de fallos. Un clasificador que siempre predice "sin fallo" alcanza 96.4% de exactitud (accuracy).

**¿Por qué la exactitud no es una métrica adecuada para este problema? ¿Qué métrica usarías en su lugar y por qué?**

*Escribe tu respuesta aquí.*

### Pregunta 2

Observa el `classification_report` generado en la sección 7. Localiza el valor de **Recall** para la clase `Failure`.

**¿Qué significa ese valor en términos del proceso industrial? Usando el número exacto de tu ejecución, ¿cuántos fallos de cada 100 reales quedarían sin detectar?**

*Escribe tu respuesta aquí.*

### Pregunta 3

Observa la gráfica de importancia de features.

**Indica las dos variables más importantes según tu ejecución. Para cada una, explica en términos físicos por qué tiene sentido que el modelo dependa de ella para predecir fallos.**

*Escribe tu respuesta aquí.*

### Pregunta 4

En el notebook del curso se usó `class_weight='balanced'` en el clasificador.

**¿Qué hace ese parámetro? ¿Por qué es necesario en este dataset?**

*Escribe tu respuesta aquí.*